<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Memoria, vectorización y optimización de pipelines con Pandas y Polars**

Este laboratorio recupera y desarrolla de forma completa el bloque específico de:

- optimización de memoria;
- vectorización;
- eliminación de operaciones innecesarias;
- reducción de copias;
- selección temprana;
- tipos eficientes;
- `apply` vs operaciones nativas;
- pipelines Pandas optimizados;
- equivalentes en Polars;
- eager vs lazy;
- validación de equivalencia.

Seguiremos el patrón del curso:

> **medir → detectar cuello de botella → visualizar → optimizar → volver a medir → validar resultados**

## Objetivos

Al finalizar deberías poder:

1. Medir memoria total y por columna.
2. Detectar dtypes sobredimensionados.
3. Identificar columnas candidatas a `category`.
4. Aplicar downcasting de forma controlada.
5. Diferenciar copias necesarias e innecesarias.
6. Comprender el coste de objetos intermedios.
7. Detectar operaciones fila a fila.
8. Sustituir `apply(axis=1)` por vectorización.
9. Sustituir bucles por expresiones nativas.
10. Reducir columnas y filas lo antes posible.
11. Construir pipelines encadenados legibles.
12. Medir tiempo y memoria antes/después.
13. Implementar el mismo pipeline en Polars.
14. Utilizar lazy execution.
15. Validar equivalencia Pandas/Polars.
16. Analizar cuándo la optimización compensa.

## 1. Preparación

In [ ]:
import time
import tracemalloc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

try:
    import polars as pl
    POLARS_DISPONIBLE = True
    print("Polars:", pl.__version__)
except ImportError:
    POLARS_DISPONIBLE = False
    print("Polars no instalado. Ejecuta: pip install polars")

print("Pandas:", pd.__version__)

# 2. Dataset suficientemente grande

In [ ]:
N = 750_000

ventas = pd.DataFrame({
    "venta_id": np.arange(1, N + 1),
    "cliente_id": rng.integers(1, 100_001, N),
    "region": rng.choice(
        ["Norte", "Sur", "Este", "Oeste", "Centro"],
        N
    ),
    "canal": rng.choice(
        ["web", "tienda", "partner", "marketplace"],
        N
    ),
    "producto": rng.choice(
        [f"P{i:03d}" for i in range(1, 101)],
        N
    ),
    "unidades": rng.integers(1, 10, N),
    "precio": rng.uniform(5, 300, N),
    "descuento": rng.choice(
        [0.0, 0.05, 0.10, 0.20],
        N,
        p=[0.55, 0.20, 0.18, 0.07]
    )
})

ventas.head()

# 3. Baseline de memoria

In [ ]:
memoria_inicial = (
    ventas.memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)

print(
    f"Memoria total: {memoria_inicial:.2f} MB"
)

## 3.1 Memoria por columna

In [ ]:
memoria_columnas = (
    ventas.memory_usage(
        deep=True
    )
    .drop("Index", errors="ignore")
    .sort_values(
        ascending=False
    )
    / 1024**2
)

memoria_columnas

## 3.2 Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    memoria_columnas.index,
    memoria_columnas.values
)

ax.set(
    title="Consumo de memoria por columna",
    xlabel="MB",
    ylabel="Columna"
)

plt.show()

### Interpretación

Las columnas de texto repetitivo suelen dominar el consumo cuando permanecen como `object`.

Antes de convertir nada, debemos revisar cardinalidad.

# 4. Cardinalidad y dtype

In [ ]:
auditoria = pd.DataFrame({
    "dtype": ventas.dtypes.astype(str),
    "n_unicos": ventas.nunique(dropna=False),
    "pct_unicos": (
        ventas.nunique(dropna=False)
        / len(ventas)
        * 100
    ),
    "memoria_MB": (
        ventas.memory_usage(deep=True)
        .drop("Index", errors="ignore")
        / 1024**2
    )
})

auditoria.sort_values(
    "memoria_MB",
    ascending=False
)

# 5. Detectar candidatas a category

In [ ]:
candidatas = auditoria.loc[
    (auditoria["dtype"] == "object")
    & (auditoria["pct_unicos"] < 5)
]

candidatas

# 6. Optimización categórica

In [ ]:
ventas_cat = ventas.copy()

for columna in [
    "region",
    "canal",
    "producto"
]:
    ventas_cat[columna] = (
        ventas_cat[columna]
        .astype("category")
    )

memoria_cat = (
    ventas_cat.memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)

print("Antes:", round(memoria_inicial, 2), "MB")
print("Después:", round(memoria_cat, 2), "MB")
print(
    "Reducción:",
    round(
        100 * (1 - memoria_cat / memoria_inicial),
        2
    ),
    "%"
)

# 7. Visualizar reducción

In [ ]:
comparacion_memoria = pd.DataFrame({
    "version": [
        "Original",
        "Categorías"
    ],
    "MB": [
        memoria_inicial,
        memoria_cat
    ]
})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    comparacion_memoria["version"],
    comparacion_memoria["MB"]
)
ax.set(
    title="Optimización mediante category",
    ylabel="Memoria (MB)"
)
plt.show()

# 8. Downcasting controlado

In [ ]:
ventas_opt = ventas_cat.copy()

for columna in [
    "venta_id",
    "cliente_id",
    "unidades"
]:
    ventas_opt[columna] = pd.to_numeric(
        ventas_opt[columna],
        downcast="unsigned"
    )

ventas_opt["precio"] = pd.to_numeric(
    ventas_opt["precio"],
    downcast="float"
)

ventas_opt["descuento"] = pd.to_numeric(
    ventas_opt["descuento"],
    downcast="float"
)

ventas_opt.dtypes

# 9. Medir impacto acumulado

In [ ]:
memoria_opt = (
    ventas_opt.memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)

memoria_versiones = pd.DataFrame({
    "version": [
        "Original",
        "Category",
        "Category + downcast"
    ],
    "MB": [
        memoria_inicial,
        memoria_cat,
        memoria_opt
    ]
})

memoria_versiones["ahorro_pct"] = (
    1
    - memoria_versiones["MB"]
    / memoria_inicial
) * 100

memoria_versiones

## 9.1 Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    memoria_versiones["version"],
    memoria_versiones["MB"]
)

ax.set(
    title="Evolución del consumo de memoria",
    ylabel="MB"
)

ax.tick_params(
    axis="x",
    rotation=15
)

plt.show()

# 10. Validar que no hemos alterado semántica

In [ ]:
validaciones = {
    "mismas_filas":
        len(ventas)
        == len(ventas_opt),

    "mismos_clientes":
        (
            ventas["cliente_id"].to_numpy()
            ==
            ventas_opt["cliente_id"].to_numpy()
        ).all(),

    "mismas_regiones":
        (
            ventas["region"].astype(str).to_numpy()
            ==
            ventas_opt["region"].astype(str).to_numpy()
        ).all(),

    "precio_aproximadamente_igual":
        np.allclose(
            ventas["precio"].to_numpy(),
            ventas_opt["precio"].to_numpy(),
            rtol=1e-5
        )
}

validaciones

### Punto importante

La optimización de dtype puede introducir pérdida de precisión.

Por eso se valida con tolerancia cuando corresponde.

# 11. Baseline de cálculo poco eficiente

In [ ]:
muestra = ventas.head(
    150_000
).copy()

def importe_apply(datos):
    salida = datos.copy()

    salida["importe"] = salida.apply(
        lambda fila:
            fila["unidades"]
            * fila["precio"]
            * (
                1
                - fila["descuento"]
            ),
        axis=1
    )

    return salida

# 12. Función de benchmark

In [ ]:
def benchmark(
    funcion,
    repeticiones=5,
    calentamiento=1
):
    for _ in range(calentamiento):
        funcion()

    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(
            time.perf_counter() - inicio
        )

    tiempos = np.array(
        tiempos
    )

    return {
        "media_ms": tiempos.mean() * 1000,
        "mediana_ms": np.median(tiempos) * 1000,
        "std_ms": tiempos.std() * 1000,
        "raw": tiempos
    }

# 13. Medir apply

In [ ]:
bench_apply = benchmark(
    lambda:
        importe_apply(muestra),
    repeticiones=5,
    calentamiento=0
)

{
    k: v
    for k, v in bench_apply.items()
    if k != "raw"
}

# 14. Vectorizar

In [ ]:
def importe_vectorizado(datos):
    salida = datos.copy()

    salida["importe"] = (
        salida["unidades"]
        * salida["precio"]
        * (
            1
            - salida["descuento"]
        )
    )

    return salida

# 15. Benchmark apply vs vectorizado

In [ ]:
bench_vec = benchmark(
    lambda:
        importe_vectorizado(
            muestra
        ),
    repeticiones=10
)

comparacion_tiempo = pd.DataFrame({
    "metodo": [
        "apply(axis=1)",
        "Vectorizado"
    ],
    "media_ms": [
        bench_apply["media_ms"],
        bench_vec["media_ms"]
    ],
    "std_ms": [
        bench_apply["std_ms"],
        bench_vec["std_ms"]
    ]
})

comparacion_tiempo

## 15.1 Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    comparacion_tiempo["metodo"],
    comparacion_tiempo["media_ms"],
    yerr=comparacion_tiempo["std_ms"],
    capsize=4
)

ax.set(
    title="apply fila a fila vs vectorización",
    ylabel="Tiempo medio (ms)"
)

plt.show()

# 16. Speedup

In [ ]:
speedup = (
    bench_apply["media_ms"]
    / bench_vec["media_ms"]
)

print(
    f"Speedup aproximado: {speedup:.2f}x"
)

# 17. Validar equivalencia

In [ ]:
resultado_apply = (
    importe_apply(
        muestra
    )["importe"]
    .to_numpy()
)

resultado_vec = (
    importe_vectorizado(
        muestra
    )["importe"]
    .to_numpy()
)

print(
    np.allclose(
        resultado_apply,
        resultado_vec
    )
)

# 18. Bucles explícitos

In [ ]:
def importe_loop(datos):
    importes = []

    for unidades, precio, descuento in zip(
        datos["unidades"],
        datos["precio"],
        datos["descuento"]
    ):
        importes.append(
            unidades
            * precio
            * (
                1
                - descuento
            )
        )

    salida = datos.copy()
    salida["importe"] = importes

    return salida

# 19. Comparar loop, apply y vectorización

In [ ]:
bench_loop = benchmark(
    lambda:
        importe_loop(muestra),
    repeticiones=5,
    calentamiento=0
)

comparacion_3 = pd.DataFrame({
    "metodo": [
        "Loop Python",
        "apply(axis=1)",
        "Vectorizado"
    ],
    "media_ms": [
        bench_loop["media_ms"],
        bench_apply["media_ms"],
        bench_vec["media_ms"]
    ]
})

comparacion_3

## 19.1 Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    comparacion_3["metodo"],
    comparacion_3["media_ms"]
)

ax.set(
    title="Tres estrategias para el mismo cálculo",
    ylabel="Tiempo medio (ms)"
)

plt.show()

### Interpretación

No todo código Python iterativo tiene el mismo coste, pero para operaciones tabulares simples la vectorización suele ser la primera alternativa que debemos probar.

# 20. Selección temprana de columnas

In [ ]:
columnas_necesarias = [
    "cliente_id",
    "region",
    "canal",
    "unidades",
    "precio",
    "descuento"
]

ventas_reducidas = (
    ventas.loc[
        :,
        columnas_necesarias
    ]
    .copy()
)

print(
    "Completo:",
    round(
        ventas.memory_usage(
            deep=True
        ).sum() / 1024**2,
        2
    ),
    "MB"
)

print(
    "Reducido:",
    round(
        ventas_reducidas.memory_usage(
            deep=True
        ).sum() / 1024**2,
        2
    ),
    "MB"
)

# 21. Filtrar temprano

In [ ]:
ventas_filtradas = (
    ventas_reducidas.loc[
        ventas_reducidas["precio"] > 100
    ]
    .copy()
)

print(
    "Filas antes:",
    len(ventas_reducidas)
)

print(
    "Filas después:",
    len(ventas_filtradas)
)

Reducir filas y columnas pronto puede disminuir el coste de todas las operaciones posteriores.

# 22. Pipeline deliberadamente ineficiente

In [ ]:
def pipeline_malo(datos):
    a = datos.copy()
    b = a.copy()

    b["importe"] = b.apply(
        lambda fila:
            fila["unidades"]
            * fila["precio"]
            * (
                1
                - fila["descuento"]
            ),
        axis=1
    )

    c = b.loc[
        b["importe"] > 500
    ].copy()

    d = c.copy()

    return (
        d.groupby(
            ["region", "canal"],
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("cliente_id", "size"),
            clientes=("cliente_id", "nunique")
        )
    )

# 23. Medir pipeline malo

In [ ]:
datos_pipeline = ventas.head(
    150_000
)

bench_pipeline_malo = benchmark(
    lambda:
        pipeline_malo(
            datos_pipeline
        ),
    repeticiones=3,
    calentamiento=0
)

bench_pipeline_malo["media_ms"]

# 24. Pipeline Pandas optimizado

In [ ]:
def pipeline_pandas_opt(datos):
    columnas = [
        "cliente_id",
        "region",
        "canal",
        "unidades",
        "precio",
        "descuento"
    ]

    resultado = (
        datos.loc[
            :,
            columnas
        ]
        .assign(
            importe=lambda x:
                x["unidades"]
                * x["precio"]
                * (
                    1
                    - x["descuento"]
                )
        )
        .loc[
            lambda x:
                x["importe"] > 500
        ]
        .groupby(
            ["region", "canal"],
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("cliente_id", "size"),
            clientes=("cliente_id", "nunique")
        )
    )

    return resultado

# 25. Medir pipeline optimizado

In [ ]:
bench_pipeline_opt = benchmark(
    lambda:
        pipeline_pandas_opt(
            datos_pipeline
        ),
    repeticiones=7
)

print(
    "Malo:",
    bench_pipeline_malo["media_ms"],
    "ms"
)

print(
    "Optimizado:",
    bench_pipeline_opt["media_ms"],
    "ms"
)

print(
    "Speedup:",
    bench_pipeline_malo["media_ms"]
    / bench_pipeline_opt["media_ms"]
)

# 26. Visualizar mejora

In [ ]:
comparacion_pipeline = pd.DataFrame({
    "version": [
        "Inicial",
        "Pandas optimizado"
    ],
    "media_ms": [
        bench_pipeline_malo["media_ms"],
        bench_pipeline_opt["media_ms"]
    ]
})

fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    comparacion_pipeline["version"],
    comparacion_pipeline["media_ms"]
)

ax.set(
    title="Optimización del pipeline Pandas",
    ylabel="Tiempo medio (ms)"
)

plt.show()

# 27. Validar equivalencia del pipeline

In [ ]:
resultado_malo = (
    pipeline_malo(
        datos_pipeline
    )
    .sort_values(
        ["region", "canal"]
    )
    .reset_index(drop=True)
)

resultado_opt = (
    pipeline_pandas_opt(
        datos_pipeline
    )
    .sort_values(
        ["region", "canal"]
    )
    .reset_index(drop=True)
)

print(
    "Facturación:",
    np.allclose(
        resultado_malo["facturacion"],
        resultado_opt["facturacion"]
    )
)

print(
    "Operaciones:",
    (
        resultado_malo["operaciones"]
        == resultado_opt["operaciones"]
    ).all()
)

# 28. Medir pico de memoria

In [ ]:
def medir_memoria(
    funcion
):
    tracemalloc.start()

    inicio = time.perf_counter()
    resultado = funcion()
    duracion = (
        time.perf_counter()
        - inicio
    )

    actual, pico = (
        tracemalloc
        .get_traced_memory()
    )

    tracemalloc.stop()

    return {
        "tiempo_ms": duracion * 1000,
        "pico_MB": pico / 1024**2,
        "resultado": resultado
    }


mem_malo = medir_memoria(
    lambda:
        pipeline_malo(
            datos_pipeline
        )
)

mem_opt = medir_memoria(
    lambda:
        pipeline_pandas_opt(
            datos_pipeline
        )
)

pd.DataFrame({
    "version": [
        "Inicial",
        "Optimizado"
    ],
    "tiempo_ms": [
        mem_malo["tiempo_ms"],
        mem_opt["tiempo_ms"]
    ],
    "pico_MB": [
        mem_malo["pico_MB"],
        mem_opt["pico_MB"]
    ]
})

# 29. Visualizar memoria

In [ ]:
comparacion_mem_pipeline = pd.DataFrame({
    "version": [
        "Inicial",
        "Optimizado"
    ],
    "pico_MB": [
        mem_malo["pico_MB"],
        mem_opt["pico_MB"]
    ]
})

fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    comparacion_mem_pipeline["version"],
    comparacion_mem_pipeline["pico_MB"]
)

ax.set(
    title="Pico de memoria del pipeline",
    ylabel="MB"
)

plt.show()

# 30. Preparar Polars

In [ ]:
if POLARS_DISPONIBLE:
    datos_pipeline_pl = pl.from_pandas(
        datos_pipeline
    )

    print(
        datos_pipeline_pl.head()
    )
else:
    print("Instalar Polars para continuar.")

# 31. Pipeline equivalente en Polars eager

In [ ]:
if POLARS_DISPONIBLE:
    def pipeline_polars(
        datos_pl
    ):
        return (
            datos_pl
            .select(
                "cliente_id",
                "region",
                "canal",
                "unidades",
                "precio",
                "descuento"
            )
            .with_columns(
                (
                    pl.col("unidades")
                    * pl.col("precio")
                    * (
                        1
                        - pl.col("descuento")
                    )
                )
                .alias("importe")
            )
            .filter(
                pl.col("importe") > 500
            )
            .group_by(
                ["region", "canal"]
            )
            .agg(
                pl.col("importe")
                .sum()
                .alias("facturacion"),

                pl.len()
                .alias("operaciones"),

                pl.col("cliente_id")
                .n_unique()
                .alias("clientes")
            )
            .sort(
                ["region", "canal"]
            )
        )

    print(
        pipeline_polars(
            datos_pipeline_pl
        )
    )

# 32. Benchmark Polars eager

In [ ]:
if POLARS_DISPONIBLE:
    bench_polars = benchmark(
        lambda:
            pipeline_polars(
                datos_pipeline_pl
            ),
        repeticiones=7
    )

    print(
        bench_polars["media_ms"],
        "ms"
    )

# 33. Pipeline Polars lazy

In [ ]:
if POLARS_DISPONIBLE:
    pipeline_lazy = (
        datos_pipeline_pl
        .lazy()
        .select(
            "cliente_id",
            "region",
            "canal",
            "unidades",
            "precio",
            "descuento"
        )
        .with_columns(
            (
                pl.col("unidades")
                * pl.col("precio")
                * (
                    1
                    - pl.col("descuento")
                )
            )
            .alias("importe")
        )
        .filter(
            pl.col("importe") > 500
        )
        .group_by(
            ["region", "canal"]
        )
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones"),

            pl.col("cliente_id")
            .n_unique()
            .alias("clientes")
        )
        .sort(
            ["region", "canal"]
        )
    )

    print(
        pipeline_lazy.explain(
            optimized=True
        )
    )

# 34. Ejecutar lazy

In [ ]:
if POLARS_DISPONIBLE:
    resultado_lazy = (
        pipeline_lazy.collect()
    )

    print(
        resultado_lazy
    )

# 35. Benchmark lazy

In [ ]:
if POLARS_DISPONIBLE:
    bench_lazy = benchmark(
        lambda:
            pipeline_lazy.collect(),
        repeticiones=7
    )

    print(
        bench_lazy["media_ms"],
        "ms"
    )

# 36. Comparación final de motores

In [ ]:
if POLARS_DISPONIBLE:
    comparacion_motores = pd.DataFrame({
        "version": [
            "Pandas inicial",
            "Pandas optimizado",
            "Polars eager",
            "Polars lazy"
        ],
        "media_ms": [
            bench_pipeline_malo["media_ms"],
            bench_pipeline_opt["media_ms"],
            bench_polars["media_ms"],
            bench_lazy["media_ms"]
        ]
    })

    comparacion_motores

## 36.1 Visualizar

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(8, 4))

    ax.bar(
        comparacion_motores["version"],
        comparacion_motores["media_ms"]
    )

    ax.set(
        title="Pipeline completo: comparación de implementaciones",
        ylabel="Tiempo medio (ms)"
    )

    ax.tick_params(
        axis="x",
        rotation=20
    )

    plt.show()

# 37. Validar Pandas vs Polars

In [ ]:
if POLARS_DISPONIBLE:
    resultado_pl_pd = (
        pipeline_polars(
            datos_pipeline_pl
        )
        .to_pandas()
        .sort_values(
            ["region", "canal"]
        )
        .reset_index(drop=True)
    )

    resultado_pd = (
        pipeline_pandas_opt(
            datos_pipeline
        )
        .sort_values(
            ["region", "canal"]
        )
        .reset_index(drop=True)
    )

    print(
        "Facturación:",
        np.allclose(
            resultado_pd["facturacion"],
            resultado_pl_pd["facturacion"]
        )
    )

    print(
        "Operaciones:",
        (
            resultado_pd["operaciones"]
            ==
            resultado_pl_pd["operaciones"]
        ).all()
    )

    print(
        "Clientes:",
        (
            resultado_pd["clientes"]
            ==
            resultado_pl_pd["clientes"]
        ).all()
    )

# 38. Coste de conversión

In [ ]:
if POLARS_DISPONIBLE:
    conversion_pd_pl = benchmark(
        lambda:
            pl.from_pandas(
                datos_pipeline
            ),
        repeticiones=7
    )

    resultado_pl_tmp = (
        pipeline_polars(
            datos_pipeline_pl
        )
    )

    conversion_pl_pd = benchmark(
        lambda:
            resultado_pl_tmp.to_pandas(),
        repeticiones=7
    )

    print(
        "Pandas → Polars:",
        round(
            conversion_pd_pl["media_ms"],
            3
        ),
        "ms"
    )

    print(
        "Polars → Pandas:",
        round(
            conversion_pl_pd["media_ms"],
            3
        ),
        "ms"
    )

### Interpretación

Si el pipeline alterna continuamente entre motores, el coste de interoperabilidad puede reducir o eliminar parte de la mejora.

La decisión correcta debe evaluarse sobre el flujo completo.

# 39. Escalabilidad

In [ ]:
tamanos = [
    10_000,
    25_000,
    50_000,
    100_000,
    150_000
]

filas_escala = []

for n in tamanos:
    datos_n = datos_pipeline.head(n)

    t_pd = benchmark(
        lambda d=datos_n:
            pipeline_pandas_opt(d),
        repeticiones=5
    )["media_ms"]

    fila = {
        "n": n,
        "Pandas": t_pd
    }

    if POLARS_DISPONIBLE:
        pl_n = pl.from_pandas(
            datos_n
        )

        t_pl = benchmark(
            lambda d=pl_n:
                pipeline_polars(d),
            repeticiones=5
        )["media_ms"]

        fila["Polars"] = t_pl

    filas_escala.append(
        fila
    )

escala = pd.DataFrame(
    filas_escala
)

escala

# 40. Visualizar escalabilidad

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    escala["n"],
    escala["Pandas"],
    marker="o",
    label="Pandas"
)

if (
    POLARS_DISPONIBLE
    and "Polars" in escala.columns
):
    ax.plot(
        escala["n"],
        escala["Polars"],
        marker="o",
        label="Polars"
    )

ax.set(
    title="Escalabilidad del pipeline optimizado",
    xlabel="Número de filas",
    ylabel="Tiempo medio (ms)"
)

ax.legend()
plt.show()

# 41. Qué NO hacer al optimizar

Evita:

- cambiar múltiples cosas sin medir cada efecto;
- convertir todos los tipos sin revisar rango;
- usar `float32` cuando la precisión importa;
- eliminar `.copy()` indiscriminadamente;
- asumir que `apply` siempre es malo;
- asumir que Polars siempre es más rápido;
- comparar motores con datasets distintos;
- incluir conversiones en un lado y no en el otro;
- optimizar antes de encontrar un cuello de botella;
- sacrificar legibilidad por mejoras irrelevantes.

# 42. Mini-práctica integrada

Partiremos de un pipeline con:

- columnas innecesarias;
- strings como `object`;
- varias copias;
- `apply(axis=1)`;
- filtro tardío;
- agregación final.

El alumnado debe producir cuatro versiones:

1. original;
2. Pandas optimizado;
3. Polars eager;
4. Polars lazy.

Y comparar:

- tiempo;
- memoria;
- resultados;
- escalabilidad.

# 43. Checklist de optimización

- [ ] ¿He medido baseline?
- [ ] ¿Sé qué columnas consumen más?
- [ ] ¿He revisado cardinalidad?
- [ ] ¿Los dtypes son adecuados?
- [ ] ¿He validado downcasting?
- [ ] ¿Hay copias redundantes?
- [ ] ¿Hay columnas que puedo descartar pronto?
- [ ] ¿Hay filas que puedo filtrar pronto?
- [ ] ¿Existe `apply(axis=1)`?
- [ ] ¿Existe un loop reemplazable?
- [ ] ¿La operación puede vectorizarse?
- [ ] ¿He medido tiempo antes/después?
- [ ] ¿He medido memoria?
- [ ] ¿He validado equivalencia?
- [ ] ¿He probado Polars?
- [ ] ¿He probado lazy cuando tiene sentido?
- [ ] ¿He medido conversiones?
- [ ] ¿He estudiado varios tamaños?

# 44. Ejercicios finales

## Ejercicio 1 — Auditoría de memoria
Calcula dtype, cardinalidad y memoria por columna.

## Ejercicio 2 — Category
Detecta automáticamente candidatas y mide el ahorro.

## Ejercicio 3 — Downcasting
Optimiza enteros y floats validando rango y precisión.

## Ejercicio 4 — Loop
Construye un loop fila a fila y vectorízalo.

## Ejercicio 5 — apply
Compara `apply(axis=1)` con expresión nativa.

## Ejercicio 6 — Copias
Construye un pipeline con tres copias y otro con una sola.

## Ejercicio 7 — Selección temprana
Descarta columnas y filas al principio del pipeline y mide el impacto.

## Ejercicio 8 — Baseline
Mide un pipeline original con al menos 10 repeticiones.

## Ejercicio 9 — Pandas optimizado
Refactoriza el pipeline manteniendo el resultado.

## Ejercicio 10 — Polars eager
Implementa el mismo pipeline.

## Ejercicio 11 — Polars lazy
Implementa el plan lazy e inspecciónalo con `explain()`.

## Ejercicio 12 — Conversiones
Mide Pandas → Polars y Polars → Pandas.

## Ejercicio 13 — Escalabilidad
Compara cinco tamaños.

## Ejercicio 14 — Memoria
Compara picos de memoria entre versiones.

## Ejercicio 15 — Reto final
Sobre al menos 750.000 filas:

1. audita memoria;
2. identifica dtypes problemáticos;
3. optimiza dtypes;
4. introduce un `apply(axis=1)`;
5. mide;
6. vectoriza;
7. elimina copias;
8. filtra temprano;
9. crea versión Pandas optimizada;
10. crea Polars eager;
11. crea Polars lazy;
12. benchmarkea;
13. mide memoria;
14. mide conversiones;
15. prueba varios tamaños;
16. valida equivalencia;
17. justifica la versión final.

# 45. Preguntas de reflexión

1. ¿Por qué `object` puede consumir mucha memoria?
2. ¿Cuándo conviene `category`?
3. ¿Qué riesgo tiene el downcasting?
4. ¿Por qué `apply(axis=1)` puede ser lento?
5. ¿Qué aporta vectorizar?
6. ¿Por qué las copias tienen coste?
7. ¿Qué significa filtrar temprano?
8. ¿Optimizar tiempo y memoria es siempre lo mismo?
9. ¿Cuándo merece la pena Polars?
10. ¿Qué aporta lazy execution?
11. ¿Por qué hay que medir conversiones?
12. ¿Cómo validar que una optimización es correcta?
13. ¿Por qué probar varios tamaños?
14. ¿Una implementación más rápida siempre es mejor?
15. ¿Qué equilibrio debe existir entre rendimiento y mantenibilidad?